In [1]:
# 导入必要的类型注解（新手可先忽略，仅用于说明类型）
from typing import Any, ClassVar

# 基础父类（简化示意，实际在 Python 内置/第三方库中）
class _AttributeHolder:
    pass

# 核心的 Namespace 类
class Namespace(_AttributeHolder):
    # 1. 构造方法：初始化对象时接收任意关键字参数
    def __init__(self, **kwargs: Any) -> None:
        # 把传入的所有关键字参数（比如 name="张三", age=18）变成对象属性
        for key, value in kwargs.items():
            self.__setattr__(key, value)

    # 2. 自定义“获取不存在的属性”的逻辑
    def __getattr__(self, name: str) -> Any:
        # 当访问 obj.xxx 但 xxx 不存在时，会触发这个方法
        raise AttributeError(f"'Namespace' object has no attribute '{name}'")

    # 3. 自定义“设置属性”的逻辑
    def __setattr__(self, name: str, value: Any, /) -> None:
        # 当执行 obj.xxx = 123 时，会触发这个方法
        # / 表示前面的参数是位置参数，不能用关键字传参（新手暂时忽略这个细节）
        super().__setattr__(name, value)  # 调用父类的设置属性方法

    # 4. 支持 "xxx in obj" 这样的判断
    def __contains__(self, key: str) -> bool:
        # 比如判断 'name' 是否在 Namespace 对象中：'name' in obj
        return hasattr(self, key)

    # 5. 支持两个 Namespace 对象用 == 比较
    def __eq__(self, other: object) -> bool:
        # 只有当 other 是 Namespace 实例，且所有属性都相等时，返回 True
        if not isinstance(other, Namespace):
            return False
        # 把对象的属性转成字典，比较字典是否相等
        return self.__dict__ == other.__dict__

    # 6. 禁用哈希（表示该对象不能作为字典的键/集合的元素）
    __hash__: ClassVar[None] = None

ns = Namespace(name="张三", age=20)
print(ns.name)  # 输出：张三
print(ns.age)   # 输出：20

张三
20


1. __init__(self, **kwargs: Any) -> None
**kwargs：是 “关键字参数” 的写法，允许你初始化时传入任意多的 key=value，比如 ns = Namespace(name="李四", score=90)。
作用：把传入的 name、score 等参数，自动变成对象的属性（ns.name、ns.score）。
ns = Namespace(name="张三", age=20)
print(ns.name)  # 输出：张三
print(ns.age)   # 输出：20

2. __getattr__(self, name: str) -> Any
触发时机：当你访问一个不存在的属性时（比如 ns.gender），Python 会自动调用这个方法。
作用：自定义 “属性不存在” 时的行为（通常是抛错，提示用户属性不存在）。
ns = Namespace(name="张三")
print(ns.gender)  # 触发 __getattr__，抛出 AttributeError: 'Namespace' object has no attribute 'gender'

In [2]:
ns = Namespace(name="张三")
print(ns.gender)  # 触发 __getattr__，抛出 AttributeError: 'Namespace' object has no attribute 'gender'

AttributeError: 'Namespace' object has no attribute 'gender'

3. __setattr__(self, name: str, value: Any, /) -> None
触发时机：当你给对象设置属性时（比如 ns.gender = "男"），Python 会自动调用这个方法。
作用：自定义 “设置属性” 的逻辑（比如校验属性名 / 值的合法性，这里是简单调用父类方法）。
新手示例：
ns = Namespace(name="张三")
ns.gender = "男"  # 触发 __setattr__，给对象新增 gender 属性
print(ns.gender)  # 输出：男

In [3]:
ns = Namespace(name="张三")
ns.gender = "男"  # 触发 __setattr__，给对象新增 gender 属性
print(ns.gender)  # 输出：男

男


4. __contains__(self, key: str) -> bool
触发时机：当你用 in 关键字判断 “某个属性是否存在” 时（比如 'name' in ns），Python 会调用这个方法。
作用：让 Namespace 对象支持 in 判断，和字典的 'key' in dict 用法一致。
新手示例：
python
运行
ns = Namespace(name="张三")
print('name' in ns)   # 输出：True
print('gender' in ns) # 输出：False

In [4]:
ns = Namespace(name="张三")
print('name' in ns)   # 输出：True
print('gender' in ns) # 输出：False

True
False


5. __eq__(self, other: object) -> bool
触发时机：当你用 == 比较两个 Namespace 对象时（比如 ns1 == ns2），Python 会调用这个方法。
作用：自定义 “相等判断” 的规则（这里是比较两个对象的所有属性是否完全一致）。
新手示例：
python
运行
ns1 = Namespace(name="张三", age=20)
ns2 = Namespace(name="张三", age=20)
ns3 = Namespace(name="李四", age=20)
print(ns1 == ns2)  # 输出：True（属性完全一致）
print(ns1 == ns3)  # 输出：False（name 不同）

In [5]:
ns1 = Namespace(name="张三", age=20)
ns2 = Namespace(name="张三", age=20)
ns3 = Namespace(name="李四", age=20)
print(ns1 == ns2)  # 输出：True（属性完全一致）
print(ns1 == ns3)  # 输出：False（name 不同）

True
False


#### 6. `__hash__: ClassVar[None]`
- **作用**：把 `__hash__` 设置为 `None`，表示这个类的实例**不能被哈希**。
- **新手理解**：哈希是字典键/集合元素的必备条件，禁用后，你不能把 `Namespace` 对象作为字典的键（`dict[ns] = 123` 会报错），也不能放入集合（`set([ns])` 会报错）—— 因为 `Namespace` 是“可变对象”（可以随时改属性），通常不适合做哈希键。

### 为什么要用 Namespace 而不是普通字典？
对新手来说，核心优势是**语法更简洁、更像“面向对象”**：
| 操作                | 字典写法          | Namespace 写法    |
|---------------------|-------------------|-------------------|
| 获取值              | `dict['name']`    | `ns.name`         |
| 设置值              | `dict['age'] = 20`| `ns.age = 20`     |
| 判断键/属性是否存在 | `'name' in dict`  | `'name' in ns`    |

### 总结
1. `Namespace` 是一个**轻量级属性容器类**，把字典的键值对变成对象的属性，用法更直观；
2. 核心魔术方法的作用：`__init__` 初始化属性、`__getattr__` 处理属性读取、`__setattr__` 处理属性设置、`__contains__` 支持 `in` 判断、`__eq__` 支持 `==` 比较；
3. 新手可以把它理解为“更易用的字典”，常用于配置管理、临时存储一组相关属性的场景（比如函数返回多个值、配置参数等）。